In [1]:
import json
import hashlib
import sys
from urllib.request import Request, build_opener, ProxyHandler

MODELO = "gemma4:31b-it-qat"
API_LOCAL = "http://127.0.0.1:11434"

# 1. Leer archivos compartidos
with open("../casos_comparacion_v1.json", "rb") as f:
    bytes_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    bytes_config = f.read()

casos = json.loads(bytes_casos)
configuracion = json.loads(bytes_config)

sha_casos = hashlib.sha256(bytes_casos).hexdigest()
sha_config = hashlib.sha256(bytes_config).hexdigest()

# 2. Verificar integridad
assert sha_casos == configuracion["cases_sha256"]

assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)

assert [c["case_id"] for c in casos] == [
    "S-005", "S-006", "S-007",
    "S-008", "S-009", "S-010"
]

assert configuracion["prompt_version"] == "V3"
assert configuracion["api_url"] == API_LOCAL + "/api/chat"

# 3. Consultar Ollama local
cliente_local = build_opener(ProxyHandler({}))

with cliente_local.open(API_LOCAL + "/api/version", timeout=10) as f:
    version_ollama = json.load(f)["version"]

with cliente_local.open(API_LOCAL + "/api/tags", timeout=10) as f:
    modelos = json.load(f)["models"]

modelo_info = next(
    (m for m in modelos if m["name"] == MODELO),
    None
)

assert modelo_info is not None, "Modelo no instalado."

# 4. Consultar capacidades y parámetros
peticion = Request(
    API_LOCAL + "/api/show",
    data=json.dumps({"model": MODELO}).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST"
)

with cliente_local.open(peticion, timeout=15) as f:
    detalles = json.load(f)

# 5. Mostrar resultados
print("Python:", sys.version.split()[0])
print("Ollama:", version_ollama)
print("Modelo:", MODELO)
print("Identificador:", modelo_info["digest"][:12])
print("Casos verificados:", len(casos))
print("SHA256:", sha_casos)
print("Prompt:", configuracion["prompt_version"])
print("Capacidades:", detalles.get("capabilities"))
print("Parámetros:", detalles.get("parameters"))

print("\nArchivos y conexión: correctos")


Python: 3.11.16
Ollama: 0.40.13
Modelo: gemma4:31b-it-qat
Identificador: e0812a55773b
Casos verificados: 6
SHA256: f73d159bb43cc9c93d2768e997052f4381a2c24bb046c82826b1793c1b8da37d
Prompt: V3
Capacidades: ['completion', 'vision', 'tools', 'thinking']
Parámetros: temperature                    1
top_k                          64
top_p                          0.95

Archivos y conexión: correctos


In [2]:
import json
import copy
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request

ARCHIVO = "resultados.jsonl"
CASO_ID = "DIAG-G4-31B-01"
CONFIG_MODELO = "gemma4_31b_qat_directa_v1"

# 1. Caso diagnóstico común
contenido = (
    "Autores: Autor X y Autor Y.\n"
    "Predicción recibida: favorable (simulada).\n"
    "Modelo predictor y puntuación: no disponibles.\n"
    "[E1] Autor X tiene un artículo sobre bases de datos académicas.\n"
    "[E2] Autor Y tiene un artículo sobre recuperación de información.\n"
    "No se proporcionan datos de colaboración previa."
)

# 2. Recuperar configuración compartida V3
solicitud = copy.deepcopy(
    configuracion["request_defaults"]
)

solicitud["model"] = MODELO

# Desactivar explícitamente el razonamiento
solicitud["think"] = False

# Configuración comparable con Gemma 4 26B A4B
solicitud["options"] = {
    "num_ctx": 4096,
    "num_predict": 2048,
    "temperature": 1,
    "top_k": 64,
    "top_p": 0.95
}

# Conservar los prompts V3 originales
solicitud["messages"] = [
    {
        "role": "system",
        "content": configuracion["system_prompt"]
    },
    {
        "role": "user",
        "content": configuracion["user_prompt_template"].format(
            case_id=CASO_ID,
            contenido=contenido
        )
    }
]

# 3. Verificaciones
assert MODELO == "gemma4:31b-it-qat"
assert configuracion["prompt_version"] == "V3"
assert "thinking" in detalles.get("capabilities", [])
assert solicitud["think"] is False
assert sha_casos == configuracion["cases_sha256"]
assert modelo_info["digest"].startswith("e0812a55773b")

# Evitar duplicar el diagnóstico
try:
    with open(ARCHIVO, encoding="utf-8") as f:
        anteriores = [
            json.loads(line)
            for line in f if line.strip()
        ]
except FileNotFoundError:
    anteriores = []

assert not any(
    r.get("case_id") == CASO_ID
    and r.get("configuracion_modelo") == CONFIG_MODELO
    for r in anteriores
), "El diagnóstico ya está registrado."

# 4. Preparar registro experimental
registro = {
    "execution_id": str(uuid4()),
    "case_id": CASO_ID,
    "grupo": "diagnostico",
    "configuracion_modelo": CONFIG_MODELO,
    "prompt_version": "V3",
    "modelo_digest": modelo_info["digest"],
    "ollama_version": version_ollama,
    "sha256_casos": sha_casos,
    "sha256_configuracion": sha_config,
    "fecha_inicio_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "solicitud": solicitud,
    "reintentos": 0
}

inicio = time.perf_counter()

# 5. Ejecutar diagnóstico en Ollama local
try:
    peticion = Request(
        configuracion["api_url"],
        data=json.dumps(
            solicitud, ensure_ascii=False
        ).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    with cliente_local.open(peticion, timeout=900) as conexion:
        respuesta = json.load(conexion)

    registro["respuesta_original"] = respuesta

    mensaje = respuesta.get("message", {})
    contenido_final = mensaje.get("content", "").strip()

    if respuesta.get("done_reason") == "length":
        registro["estado"] = "truncada"

    elif mensaje.get("thinking"):
        registro["estado"] = "revisar_modo"

    elif (
        respuesta.get("done_reason") == "stop"
        and contenido_final
    ):
        registro["estado"] = "ok"

    else:
        registro["estado"] = "revisar"

except Exception as error:
    registro["estado"] = "error"
    registro["error_tipo"] = type(error).__name__
    registro["error"] = str(error)

registro["tiempo_cliente_s"] = round(
    time.perf_counter() - inicio, 3
)

registro["fecha_fin_utc"] = datetime.now(
    timezone.utc
).isoformat()

# 6. Guardar resultado sin sobrescribir
with open(ARCHIVO, "a", encoding="utf-8") as f:
    f.write(
        json.dumps(registro, ensure_ascii=False) + "\n"
    )

# 7. Mostrar métricas y respuesta final
print("Caso:", CASO_ID)
print("Configuración:", CONFIG_MODELO)
print("Estado:", registro["estado"])
print("Tiempo cliente:", registro["tiempo_cliente_s"], "s")

if "respuesta_original" in registro:
    r = registro["respuesta_original"]
    mensaje = r.get("message", {})

    print("Tokens entrada:", r.get("prompt_eval_count"))
    print("Tokens generados:", r.get("eval_count"))
    print("Terminación:", r.get("done_reason"))
    print("Razonamiento separado:", bool(
        mensaje.get("thinking")
    ))

    for campo in (
        "total_duration",
        "load_duration",
        "prompt_eval_duration",
        "eval_duration"
    ):
        valor = r.get(campo)

        if valor is not None:
            print(campo, round(valor / 1e9, 3), "s")

    print("\nRESPUESTA FINAL:\n")
    print(mensaje.get("content", ""))

else:
    print("ERROR:", registro.get("error"))

print("\nRegistro guardado en resultados.jsonl")


Caso: DIAG-G4-31B-01
Configuración: gemma4_31b_qat_directa_v1
Estado: ok
Tiempo cliente: 100.569 s
Tokens entrada: 370
Tokens generados: 244
Terminación: stop
Razonamiento separado: False
total_duration 100.567 s
load_duration 16.669 s
prompt_eval_duration 1.882 s
eval_duration 82.006 s

RESPUESTA FINAL:

**Evidencia**
El Autor X ha publicado un trabajo referente a bases de datos académicas [E1]. El Autor Y ha publicado un trabajo centrado en la recuperación de información [E2].

**Interpretación**
Existe una convergencia temática entre las líneas de investigación de ambos autores, ya que el estudio de las bases de datos académicas [E1] y la recuperación de información [E2] constituyen áreas complementarias dentro del ámbito de la gestión de la información. Esta coincidencia en el dominio del conocimiento respalda la pertinencia académica de la colaboración sugerida.

**Limitaciones**
La evidencia proporcionada es insuficiente para explicar el funcionamiento interno del predictor ni el

In [3]:
import json
import copy
import hashlib
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request, build_opener, ProxyHandler

MODELO = "gemma4:31b-it-qat"
API = "http://127.0.0.1:11434"
ARCHIVO = "resultados.jsonl"
CONFIG_MODELO = "gemma4_31b_qat_directa_v1"

# 1. Verificar archivos compartidos
with open("../casos_comparacion_v1.json", "rb") as f:
    bytes_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    bytes_config = f.read()

casos = json.loads(bytes_casos)
config = json.loads(bytes_config)

sha_casos = hashlib.sha256(bytes_casos).hexdigest()
sha_config = hashlib.sha256(bytes_config).hexdigest()

assert sha_casos == config["cases_sha256"]

assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)

assert [c["case_id"] for c in casos] == [
    "S-005", "S-006", "S-007",
    "S-008", "S-009", "S-010"
]

assert config["prompt_version"] == "V3"
assert config["api_url"] == API + "/api/chat"

# 2. Recuperar diagnóstico exitoso
with open(ARCHIVO, encoding="utf-8") as f:
    anteriores = [
        json.loads(line)
        for line in f if line.strip()
    ]

diagnostico = next(
    r for r in anteriores
    if r["case_id"] == "DIAG-G4-31B-01"
    and r["configuracion_modelo"] == CONFIG_MODELO
)

assert diagnostico["estado"] == "ok"
assert diagnostico["solicitud"]["model"] == MODELO
assert diagnostico["solicitud"]["think"] is False

assert diagnostico["solicitud"]["options"] == {
    "num_ctx": 4096,
    "num_predict": 2048,
    "temperature": 1,
    "top_k": 64,
    "top_p": 0.95
}

assert diagnostico["sha256_casos"] == sha_casos
assert diagnostico["sha256_configuracion"] == sha_config
assert diagnostico["solicitud"]["messages"][0]["content"] == (
    config["system_prompt"]
)

# 3. Verificar identidad del modelo instalado
cliente = build_opener(ProxyHandler({}))

with cliente.open(API + "/api/tags", timeout=10) as f:
    modelos = json.load(f)["models"]

modelo_info = next(
    m for m in modelos if m["name"] == MODELO
)

assert modelo_info["digest"] == diagnostico["modelo_digest"]

# Evitar duplicar ejecuciones anteriores
registrados = {
    r["case_id"] for r in anteriores
    if r.get("configuracion_modelo") == CONFIG_MODELO
    and r.get("grupo") == "comparacion"
}

# 4. Ejecutar los seis casos
with open(ARCHIVO, "a", encoding="utf-8") as archivo:

    for caso in casos:

        case_id = caso["case_id"]

        if case_id in registrados:
            print(case_id, "| ya registrado", flush=True)
            continue

        # Recuperar solicitud exacta del diagnóstico
        solicitud = copy.deepcopy(
            diagnostico["solicitud"]
        )

        # Sustituir solamente el caso y sus evidencias
        solicitud["messages"][1]["content"] = (
            config["user_prompt_template"].format(
                case_id=case_id,
                contenido=caso["contenido"]
            )
        )

        registro = {
            "execution_id": str(uuid4()),
            "case_id": case_id,
            "tipo": caso["tipo"],
            "grupo": "comparacion",
            "configuracion_modelo": CONFIG_MODELO,
            "prompt_version": "V3",
            "modelo_digest": modelo_info["digest"],
            "ollama_version": diagnostico["ollama_version"],
            "sha256_casos": sha_casos,
            "sha256_configuracion": sha_config,
            "fecha_inicio_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "solicitud": solicitud,
            "reintentos": 0
        }

        print("Ejecutando:", case_id, flush=True)
        inicio = time.perf_counter()

        # 5. Ejecutar inferencia en Ollama
        try:
            peticion = Request(
                config["api_url"],
                data=json.dumps(
                    solicitud, ensure_ascii=False
                ).encode("utf-8"),
                headers={"Content-Type": "application/json"},
                method="POST"
            )

            with cliente.open(peticion, timeout=900) as conexion:
                respuesta = json.load(conexion)

            registro["respuesta_original"] = respuesta

            mensaje = respuesta.get("message", {})
            contenido_final = mensaje.get("content", "").strip()

            if respuesta.get("done_reason") == "length":
                registro["estado"] = "truncada"

            elif mensaje.get("thinking"):
                registro["estado"] = "revisar_modo"

            elif (
                respuesta.get("done_reason") == "stop"
                and contenido_final
            ):
                registro["estado"] = "ok"

            else:
                registro["estado"] = "revisar"

        except Exception as error:
            registro["estado"] = "error"
            registro["error_tipo"] = type(error).__name__
            registro["error"] = str(error)

        registro["tiempo_cliente_s"] = round(
            time.perf_counter() - inicio, 3
        )

        registro["fecha_fin_utc"] = datetime.now(
            timezone.utc
        ).isoformat()

        # 6. Guardar cada ejecución inmediatamente
        archivo.write(
            json.dumps(registro, ensure_ascii=False) + "\n"
        )
        archivo.flush()

        r = registro.get("respuesta_original", {})

        print(
            case_id,
            "|", registro["estado"],
            "|", r.get("eval_count", "-"), "tokens",
            "|", registro["tiempo_cliente_s"], "s",
            flush=True
        )

print("\nArchivo actualizado:", ARCHIVO)


Ejecutando: S-005
S-005 | ok | 309 tokens | 108.43 s
Ejecutando: S-006
S-006 | ok | 256 tokens | 88.627 s
Ejecutando: S-007
S-007 | ok | 291 tokens | 100.965 s
Ejecutando: S-008
S-008 | ok | 280 tokens | 96.616 s
Ejecutando: S-009
S-009 | ok | 274 tokens | 94.563 s
Ejecutando: S-010
S-010 | ok | 304 tokens | 104.493 s

Archivo actualizado: resultados.jsonl
